# Instruction Dataset Generation for QLoRA Fine-Tuning

Pulls chunks from our Snowflake arXiv corpus and uses Gemini to generate gold-standard outputs across 5 research paper task types.

**Output:** `instruction_dataset.json` (30-50 examples for QLoRA training)

## Cell 1: Install Dependencies

In [ ]:
!pip install snowflake-connector-python google-genai python-dotenv

## Cell 2: Imports and Configuration

In [ ]:
import json
import os
import random
import time
from pathlib import Path

import snowflake.connector
from google import genai
from dotenv import load_dotenv

load_dotenv()

# ── Configuration ────────────────────────────────────────────
GEMINI_MODEL = "gemini-2.5-flash"
EXAMPLES_PER_TASK = 8          # aim for 6-10 usable per task after review
OUTPUT_PATH = "../data/instruction_dataset.json"

# Snowflake session context
SF_WAREHOUSE = "ROHAN_BLAKE_KENNETH_WH"
SF_DATABASE = "CS5542_PROJECT_ROHAN_BLAKE_KENNETH"

## Cell 3: Connect to Snowflake

In [ ]:
conn = snowflake.connector.connect(
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    warehouse=SF_WAREHOUSE,
    database=SF_DATABASE,
    role=os.getenv("SNOWFLAKE_ROLE", None),
)

cur = conn.cursor()
cur.execute(f"USE WAREHOUSE {SF_WAREHOUSE}")
cur.execute(f"USE DATABASE {SF_DATABASE}")
cur.execute("SELECT CURRENT_USER(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()")
print("Connected:", cur.fetchone())

## Cell 4: Initialize Gemini Client

In [ ]:
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# Quick test
test = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Say hello in one word.",
)
print("Gemini OK:", test.text)

## Cell 5: Define Task Types and Gemini Prompts

In [ ]:
TASKS = {
    "summarize_paper": {
        "instruction": "Summarize the key contributions and findings of this research paper excerpt.",
        "gemini_prompt": (
            "Read the following research paper excerpt and write a concise 3-5 sentence summary "
            "of its key contributions and findings. Be specific about methods and results mentioned. "
            "Do not start with 'This paper' or 'This excerpt' — vary your opening."
        ),
        "preferred_sections": ["abstract", "introduction", "conclusion"],
        "chunks_needed": 1,
    },
    "compare_methods": {
        "instruction": "Compare the methodologies described in these two research excerpts.",
        "gemini_prompt": (
            "Read the following two research paper excerpts and write a 3-5 sentence comparison "
            "of their methodologies. Identify similarities, differences, and relative strengths. "
            "Be specific about the techniques and approaches each paper uses."
        ),
        "preferred_sections": ["introduction", "method", "approach", "model"],
        "chunks_needed": 2,
    },
    "extract_results": {
        "instruction": "Extract the key experimental results and findings from this research excerpt.",
        "gemini_prompt": (
            "Read the following research paper excerpt and extract all quantitative results, "
            "metrics, baselines, and key findings. Present them in a structured format with "
            "bullet points. Include specific numbers where available."
        ),
        "preferred_sections": ["results", "experiments", "evaluation", "discussion"],
        "chunks_needed": 1,
    },
    "identify_limitations": {
        "instruction": "Identify the limitations and potential weaknesses in this research.",
        "gemini_prompt": (
            "Read the following research paper excerpt and identify any limitations, weaknesses, "
            "assumptions, or gaps. Include both limitations the authors acknowledge and any you "
            "can reasonably infer from the text. Write 3-5 specific points."
        ),
        "preferred_sections": ["discussion", "conclusion", "method", "approach"],
        "chunks_needed": 1,
    },
    "suggest_related_work": {
        "instruction": "Based on this research excerpt, suggest related research directions and adjacent fields worth exploring.",
        "gemini_prompt": (
            "Read the following research paper excerpt and suggest 3-5 related research directions, "
            "adjacent fields, or follow-up studies that would build on this work. Be specific about "
            "what each direction would investigate and why it follows from this work."
        ),
        "preferred_sections": ["abstract", "introduction", "conclusion", "related work"],
        "chunks_needed": 1,
    },
}

print(f"Defined {len(TASKS)} task types")
for name, cfg in TASKS.items():
    print(f"  {name}: {cfg['chunks_needed']} chunk(s) per example, sections={cfg['preferred_sections']}")

## Cell 6: Pull Chunks from Snowflake

Pulls a large pool of candidate chunks, then filters for quality.

In [ ]:
cur.execute(f"USE WAREHOUSE {SF_WAREHOUSE}")
cur.execute(f"USE DATABASE {SF_DATABASE}")

cur.execute("""
    SELECT
        c.CHUNK_ID,
        c.PAPER_ID,
        c.SECTION_NAME,
        c.TEXT_CONTENT,
        c.WORD_COUNT,
        p.TITLE,
        p.ABSTRACT
    FROM RAW.CHUNKS c
    JOIN RAW.PAPERS p ON c.PAPER_ID = p.PAPER_ID
    WHERE c.WORD_COUNT BETWEEN 80 AND 250
    ORDER BY RANDOM()
    LIMIT 500
""")

columns = [desc[0] for desc in cur.description]
all_chunks = [dict(zip(columns, row)) for row in cur.fetchall()]

print(f"Pulled {len(all_chunks)} candidate chunks")

# Show section distribution
from collections import Counter
section_counts = Counter(c["SECTION_NAME"] for c in all_chunks)
print("\nSection distribution:")
for section, count in section_counts.most_common(15):
    print(f"  {section}: {count}")

## Cell 7: Filter and Select Chunks per Task Type

Match chunks to tasks based on their section. Ensure diversity across papers.

In [ ]:
def section_matches(section_name, preferred_sections):
    """Check if a chunk's section matches any of the preferred sections (fuzzy)."""
    if not section_name:
        return False
    section_lower = section_name.lower()
    return any(pref in section_lower for pref in preferred_sections)


def is_quality_chunk(chunk):
    """Filter out low-quality chunks."""
    text = chunk["TEXT_CONTENT"]
    # Skip chunks that are mostly references or citations
    if text.count("[") > 10:
        return False
    # Skip chunks that are mostly equations
    if text.count("=") > 8:
        return False
    # Skip acknowledgments/funding
    lower = text.lower()
    if any(kw in lower for kw in ["acknowledgment", "funding", "supported by grant", "nsf grant"]):
        return False
    return True


# Filter for quality
quality_chunks = [c for c in all_chunks if is_quality_chunk(c)]
print(f"{len(quality_chunks)} chunks passed quality filter (dropped {len(all_chunks) - len(quality_chunks)})")

# Select chunks per task type
selected = {}
used_paper_ids = set()  # track to encourage diversity

for task_name, cfg in TASKS.items():
    # Prefer section-matched chunks, fall back to any quality chunk
    matched = [c for c in quality_chunks if section_matches(c["SECTION_NAME"], cfg["preferred_sections"])]
    fallback = [c for c in quality_chunks if c not in matched]
    pool = matched + fallback

    # Shuffle and pick, preferring chunks from papers we haven't used yet
    random.shuffle(pool)
    pool.sort(key=lambda c: c["PAPER_ID"] in used_paper_ids)

    if cfg["chunks_needed"] == 1:
        picks = pool[:EXAMPLES_PER_TASK]
        selected[task_name] = picks
        for p in picks:
            used_paper_ids.add(p["PAPER_ID"])
    else:
        # For compare_methods: pick pairs from different papers
        pairs = []
        used_in_pairs = set()
        for i, c1 in enumerate(pool):
            if c1["CHUNK_ID"] in used_in_pairs:
                continue
            for c2 in pool[i + 1:]:
                if c2["CHUNK_ID"] in used_in_pairs:
                    continue
                if c1["PAPER_ID"] != c2["PAPER_ID"]:  # different papers
                    pairs.append((c1, c2))
                    used_in_pairs.update([c1["CHUNK_ID"], c2["CHUNK_ID"]])
                    used_paper_ids.update([c1["PAPER_ID"], c2["PAPER_ID"]])
                    break
            if len(pairs) >= EXAMPLES_PER_TASK:
                break
        selected[task_name] = pairs

print("\nSelected chunks per task:")
for task_name, items in selected.items():
    print(f"  {task_name}: {len(items)} examples")
print(f"\nUsing chunks from {len(used_paper_ids)} distinct papers")

## Cell 8: Preview Selected Chunks

Sanity check — make sure the chunks look reasonable before sending to Gemini.

In [ ]:
for task_name, items in selected.items():
    print(f"\n{'='*60}")
    print(f"Task: {task_name}")
    print(f"{'='*60}")
    for i, item in enumerate(items[:2]):  # preview first 2 per task
        if isinstance(item, tuple):
            c1, c2 = item
            print(f"\n  Example {i+1} (pair):")
            print(f"    Paper 1: {c1['TITLE'][:80]}")
            print(f"    Section: {c1['SECTION_NAME']}")
            print(f"    Text:    {c1['TEXT_CONTENT'][:150]}...")
            print(f"    Paper 2: {c2['TITLE'][:80]}")
            print(f"    Section: {c2['SECTION_NAME']}")
            print(f"    Text:    {c2['TEXT_CONTENT'][:150]}...")
        else:
            print(f"\n  Example {i+1}:")
            print(f"    Paper:   {item['TITLE'][:80]}")
            print(f"    Section: {item['SECTION_NAME']}")
            print(f"    Text:    {item['TEXT_CONTENT'][:150]}...")

## Cell 9: Generate Gold-Standard Outputs with Gemini

This cell takes a few minutes — it makes one Gemini API call per example.

In [ ]:
dataset = []
errors = []

for task_name, cfg in TASKS.items():
    items = selected[task_name]
    print(f"\nGenerating {len(items)} examples for: {task_name}")

    for i, item in enumerate(items):
        # Build the input text
        if isinstance(item, tuple):
            c1, c2 = item
            input_text = (
                f"Excerpt 1 (from \"{c1['TITLE']}\"):\n{c1['TEXT_CONTENT']}\n\n"
                f"Excerpt 2 (from \"{c2['TITLE']}\"):\n{c2['TEXT_CONTENT']}"
            )
        else:
            input_text = item["TEXT_CONTENT"]

        # Build the Gemini prompt
        gemini_prompt = f"{cfg['gemini_prompt']}\n\n{input_text}"

        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=gemini_prompt,
            )
            output_text = response.text.strip()

            dataset.append({
                "task_type": task_name,
                "instruction": cfg["instruction"],
                "input": input_text,
                "output": output_text,
            })
            print(f"  [{i+1}/{len(items)}] OK ({len(output_text)} chars)")

        except Exception as e:
            print(f"  [{i+1}/{len(items)}] ERROR: {e}")
            errors.append({"task": task_name, "index": i, "error": str(e)})

        # Small delay to avoid rate limits
        time.sleep(1)

print(f"\nGenerated {len(dataset)} examples total")
if errors:
    print(f"Errors: {len(errors)}")
    for e in errors:
        print(f"  {e}")

## Cell 10: Review Generated Examples

Print all examples for manual review. Flag anything that looks wrong.

In [ ]:
for i, ex in enumerate(dataset):
    print(f"\n{'='*60}")
    print(f"Example {i} | task_type: {ex['task_type']}")
    print(f"{'='*60}")
    print(f"INSTRUCTION: {ex['instruction']}")
    print(f"INPUT ({len(ex['input'])} chars): {ex['input'][:300]}...")
    print(f"OUTPUT ({len(ex['output'])} chars):")
    print(ex["output"])

# Distribution check
print(f"\n{'='*60}")
print("Distribution:")
dist = Counter(ex["task_type"] for ex in dataset)
for task, count in dist.most_common():
    print(f"  {task}: {count}")
print(f"  TOTAL: {len(dataset)}")

## Cell 11: Manual Corrections

After reviewing the examples above, drop bad ones and fix outputs here.

In [ ]:
# ── Drop bad examples by index ──────────────────────────────
# Fill these in after reviewing Cell 10 output
drop_indices = [
    # e.g. 3, 17, 28
]

if drop_indices:
    dataset = [ex for i, ex in enumerate(dataset) if i not in drop_indices]
    print(f"Dropped {len(drop_indices)} examples, {len(dataset)} remaining")

# ── Fix specific outputs ────────────────────────────────────
# Uncomment and edit as needed after review
#
# dataset[5]["output"] = "corrected output here..."
# dataset[12]["output"] = "corrected output here..."

# Final distribution
dist = Counter(ex["task_type"] for ex in dataset)
print("\nFinal distribution:")
for task, count in dist.most_common():
    print(f"  {task}: {count}")
print(f"  TOTAL: {len(dataset)}")

## Cell 12: Save Dataset

In [ ]:
output_path = Path(OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f"Saved {len(dataset)} examples to {output_path.resolve()}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

# Close Snowflake connection
conn.close()
print("Snowflake connection closed.")